# Preprocesamiento de Datos
## Proyecto Final - MIAA 2025 - Universidad ICESI

Este notebook contiene todos los pasos de preprocesamiento de datos necesarios para preparar los datos para el entrenamiento de modelos de machine learning.

## 1. Configuración e Importación de Librerías

In [ ]:
# Importación de librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings

# Importar módulos del proyecto
import sys
sys.path.append('../src')
from data_preprocessing import DataPreprocessor, generate_sample_data
from utils import basic_info, save_results

# Configuración
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')
pd.set_option('display.max_columns', None)

print("✅ Librerías y módulos importados correctamente")

## 2. Carga de Datos

In [ ]:
# Cargar datos (usar datos de ejemplo o cargar desde archivo)
# Para este ejemplo, generaremos datos de muestra
df_raw = generate_sample_data(n_samples=1000)

print(f"✅ Datos cargados: {df_raw.shape[0]} filas, {df_raw.shape[1]} columnas")
print("\nPrimeras 5 filas:")
df_raw.head()

In [ ]:
# Información básica de los datos originales
print("=== INFORMACIÓN DE DATOS ORIGINALES ===")
basic_info(df_raw)

## 3. Inicialización del Preprocesador

In [ ]:
# Crear instancia del preprocesador
preprocessor = DataPreprocessor()

# Hacer una copia de los datos para procesamiento
df = df_raw.copy()

print("✅ Preprocesador inicializado")
print(f"Forma inicial del dataset: {df.shape}")

## 4. Manejo de Valores Faltantes

In [ ]:
# Análizar valores faltantes antes del procesamiento
print("=== VALORES FALTANTES ANTES DEL PROCESAMIENTO ===")
missing_before = df.isnull().sum()
missing_before = missing_before[missing_before > 0]

if len(missing_before) > 0:
    print(missing_before)
    
    # Visualizar valores faltantes
    plt.figure(figsize=(10, 6))
    sns.heatmap(df.isnull(), yticklabels=False, cbar=True, cmap='viridis')
    plt.title('Patrón de Valores Faltantes - Antes del Procesamiento')
    plt.tight_layout()
    plt.show()
else:
    print("No se encontraron valores faltantes")

In [ ]:
# Imputar valores faltantes
df_imputed = preprocessor.handle_missing_values(df, strategy='mean')

print("=== VALORES FALTANTES DESPUÉS DE LA IMPUTACIÓN ===")
missing_after = df_imputed.isnull().sum()
missing_after = missing_after[missing_after > 0]

if len(missing_after) > 0:
    print(missing_after)
else:
    print("✅ Todos los valores faltantes han sido imputados")

# Actualizar el dataframe principal
df = df_imputed.copy()
print(f"Forma del dataset después de imputación: {df.shape}")

## 5. Detección y Manejo de Outliers

In [ ]:
# Identificar variables numéricas para análisis de outliers
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Variables numéricas para análisis de outliers: {numeric_columns}")

# Visualizar outliers antes del procesamiento
if len(numeric_columns) > 0:
    fig, axes = plt.subplots(2, len(numeric_columns), figsize=(15, 10))
    
    for i, col in enumerate(numeric_columns):
        # Boxplot
        if len(numeric_columns) > 1:
            axes[0, i].boxplot(df[col])
            axes[0, i].set_title(f'Boxplot - {col}')
            
            # Histograma
            axes[1, i].hist(df[col], bins=30, alpha=0.7)
            axes[1, i].set_title(f'Histograma - {col}')
        else:
            axes[0].boxplot(df[col])
            axes[0].set_title(f'Boxplot - {col}')
            
            axes[1].hist(df[col], bins=30, alpha=0.7)
            axes[1].set_title(f'Histograma - {col}')
    
    plt.suptitle('Análisis de Outliers - Antes del Procesamiento')
    plt.tight_layout()
    plt.show()

In [ ]:
# Remover outliers (opcional - comentar si no se desea)
print(f"Forma antes de remover outliers: {df.shape}")

# Remover outliers usando método IQR
df_no_outliers = preprocessor.remove_outliers(df, columns=numeric_columns, method='iqr', threshold=1.5)

print(f"Forma después de remover outliers: {df_no_outliers.shape}")
print(f"Filas removidas: {df.shape[0] - df_no_outliers.shape[0]}")

# Decidir si usar datos con o sin outliers
use_no_outliers = input("¿Usar datos sin outliers? (y/n): ").lower().strip()
if use_no_outliers == 'y':
    df = df_no_outliers.copy()
    print("✅ Usando dataset sin outliers")
else:
    print("✅ Manteniendo outliers en el dataset")

## 6. Codificación de Variables Categóricas

In [ ]:
# Identificar variables categóricas
categorical_columns = df.select_dtypes(include=['object']).columns.tolist()
print(f"Variables categóricas: {categorical_columns}")

if len(categorical_columns) > 0:
    # Mostrar valores únicos de cada variable categórica
    for col in categorical_columns:
        unique_values = df[col].nunique()
        print(f"{col}: {unique_values} valores únicos")
        print(f"  Valores: {df[col].unique()[:10]}...")  # Mostrar solo los primeros 10
        print()

In [ ]:
# Codificar variables categóricas
if len(categorical_columns) > 0:
    print("Codificando variables categóricas usando One-Hot Encoding...")
    df_encoded = preprocessor.encode_categorical_variables(df, categorical_columns, method='onehot')
    
    print(f"Forma después de codificación: {df_encoded.shape}")
    print(f"Nuevas columnas creadas: {df_encoded.shape[1] - df.shape[1] + len(categorical_columns)}")
    
    # Mostrar nuevas columnas
    new_columns = [col for col in df_encoded.columns if col not in df.columns or col in categorical_columns]
    print(f"Nuevas columnas: {new_columns[:10]}...")  # Mostrar solo las primeras 10
    
    # Actualizar dataframe
    df = df_encoded.copy()
    print("✅ Variables categóricas codificadas")
else:
    print("No hay variables categóricas para codificar")

## 7. Escalado de Características

In [ ]:
# Identificar columnas numéricas para escalado
numeric_columns_final = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Variables numéricas para escalado: {len(numeric_columns_final)} columnas")

# Mostrar estadísticas antes del escalado
print("\n=== ESTADÍSTICAS ANTES DEL ESCALADO ===")
print(df[numeric_columns_final].describe())

In [ ]:
# Aplicar escalado estándar
df_scaled = preprocessor.scale_features(df, numeric_columns_final, method='standard')

print("\n=== ESTADÍSTICAS DESPUÉS DEL ESCALADO ===")
print(df_scaled[numeric_columns_final].describe())

# Visualizar distribuciones antes y después del escalado
fig, axes = plt.subplots(2, min(4, len(numeric_columns_final)), figsize=(16, 8))

for i, col in enumerate(numeric_columns_final[:4]):  # Mostrar solo las primeras 4
    if len(numeric_columns_final) > 1:
        # Antes del escalado
        axes[0, i].hist(df[col], bins=30, alpha=0.7, color='skyblue')
        axes[0, i].set_title(f'{col} - Original')
        
        # Después del escalado
        axes[1, i].hist(df_scaled[col], bins=30, alpha=0.7, color='lightcoral')
        axes[1, i].set_title(f'{col} - Escalado')
    else:
        axes[0].hist(df[col], bins=30, alpha=0.7, color='skyblue')
        axes[0].set_title(f'{col} - Original')
        
        axes[1].hist(df_scaled[col], bins=30, alpha=0.7, color='lightcoral')
        axes[1].set_title(f'{col} - Escalado')
        break

plt.suptitle('Comparación: Antes vs Después del Escalado')
plt.tight_layout()
plt.show()

# Actualizar dataframe
df = df_scaled.copy()
print("✅ Características escaladas")

## 8. División de Datos (Train/Test Split)

In [ ]:
# Para este ejemplo, usaremos una variable como target
# En un caso real, esto debería ser tu variable objetivo específica
target_column = 'score'  # Cambiar según tu caso específico

# Verificar si la columna target existe
if target_column in df.columns:
    # Separar características (X) y variable objetivo (y)
    X = df.drop(columns=[target_column])
    y = df[target_column]
    
    print(f"Características (X): {X.shape}")
    print(f"Variable objetivo (y): {y.shape}")
    print(f"Tipo de variable objetivo: {y.dtype}")
else:
    print(f"⚠️  Columna '{target_column}' no encontrada. Usando todas las columnas como características.")
    X = df
    # Crear una variable objetivo sintética para demostración
    y = pd.Series(np.random.randint(0, 2, len(df)), name='synthetic_target')
    print(f"Características (X): {X.shape}")
    print(f"Variable objetivo sintética (y): {y.shape}")

In [ ]:
# Crear división train/test
X_train, X_test, y_train, y_test = preprocessor.create_train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("=== DIVISIÓN TRAIN/TEST COMPLETADA ===")
print(f"Conjunto de entrenamiento (X_train): {X_train.shape}")
print(f"Conjunto de prueba (X_test): {X_test.shape}")
print(f"Etiquetas de entrenamiento (y_train): {y_train.shape}")
print(f"Etiquetas de prueba (y_test): {y_test.shape}")

# Verificar distribución de la variable objetivo
if y.dtype == 'object' or y.nunique() < 10:
    print("\n=== DISTRIBUCIÓN DE LA VARIABLE OBJETIVO ===")
    print("Entrenamiento:")
    print(y_train.value_counts().sort_index())
    print("\nPrueba:")
    print(y_test.value_counts().sort_index())
else:
    print("\n=== ESTADÍSTICAS DE LA VARIABLE OBJETIVO ===")
    print("Entrenamiento:")
    print(y_train.describe())
    print("\nPrueba:")
    print(y_test.describe())

## 9. Resumen del Preprocesamiento

In [ ]:
print("=== RESUMEN DEL PREPROCESAMIENTO ===")
print(f"\n📊 TRANSFORMACIONES APLICADAS:")
print(f"   ✅ Imputación de valores faltantes")
print(f"   ✅ Codificación de variables categóricas (One-Hot)")
print(f"   ✅ Escalado de características (StandardScaler)")
print(f"   ✅ División train/test (80/20)")
if 'use_no_outliers' in locals() and use_no_outliers == 'y':
    print(f"   ✅ Remoción de outliers")

print(f"\n📈 DATOS FINALES:")
print(f"   • Dataset original: {df_raw.shape[0]} filas, {df_raw.shape[1]} columnas")
print(f"   • Dataset procesado: {X.shape[0]} filas, {X.shape[1]} características")
print(f"   • Conjunto de entrenamiento: {X_train.shape[0]} muestras")
print(f"   • Conjunto de prueba: {X_test.shape[0]} muestras")

print(f"\n🎯 VARIABLE OBJETIVO:")
if target_column in df_raw.columns:
    if y.dtype == 'object' or y.nunique() < 10:
        print(f"   • Tipo: Clasificación")
        print(f"   • Clases: {y.nunique()}")
        print(f"   • Distribución: {dict(y.value_counts())}")
    else:
        print(f"   • Tipo: Regresión")
        print(f"   • Rango: [{y.min():.2f}, {y.max():.2f}]")
        print(f"   • Media: {y.mean():.2f}")
else:
    print(f"   • Variable objetivo sintética creada para demostración")

print(f"\n✅ ¡Datos listos para entrenamiento de modelos!")

## 10. Guardado de Datos Procesados y Preprocesadores

In [ ]:
# Crear directorio para datos procesados si no existe
import os
os.makedirs('../data/processed', exist_ok=True)

# Guardar conjuntos de entrenamiento y prueba
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print("✅ Conjuntos de datos guardados en '../data/processed/'")

# Guardar preprocesadores entrenados
preprocessor.save_preprocessors('../models/preprocessors.joblib')

# Guardar información del procesamiento
processing_info = {
    'original_shape': df_raw.shape,
    'processed_shape': X.shape,
    'train_shape': X_train.shape,
    'test_shape': X_test.shape,
    'target_column': target_column,
    'categorical_columns': categorical_columns,
    'numeric_columns': numeric_columns_final,
    'preprocessing_steps': [
        'handle_missing_values',
        'encode_categorical_variables',
        'scale_features',
        'train_test_split'
    ]
}

save_results(processing_info, 'preprocessing_info.csv')
print("✅ Información de procesamiento guardada")

print("\n🎉 ¡Preprocesamiento completado exitosamente!")
print("📁 Archivos generados:")
print("   • ../data/processed/X_train.csv")
print("   • ../data/processed/X_test.csv")
print("   • ../data/processed/y_train.csv")
print("   • ../data/processed/y_test.csv")
print("   • ../models/preprocessors.joblib")
print("   • ../results/preprocessing_info.csv")